# Tokenization Study

## Datasets: Loading and Cleaning

### Itihasa

In [ ]:
import requests

url = "https://raw.githubusercontent.com/rahular/itihasa/main/data/dev.sn"

data = requests.get(url).text.splitlines()

In [ ]:
data[:2]

In [ ]:
# searching for Sandhis: This just means that this dataset has sandhis and is not a disolved dataset.

matches = [line for line in data if "ऽ" in line]

print(len(matches))
print(matches[0])

In [ ]:
import re

def clean_text(corpus:list):

    texts = []

    for sentence in corpus:
      sentence = sentence.strip()

      # normalize whitespace
      sentence = re.sub(r"\s+", " ", sentence)
      sentence = re.sub(r"\.", "", sentence)

      # separate danda symbols
      sentence = re.sub(r"॥", " ॥ ", sentence)
      sentence = re.sub(r"।", " । ", sentence)

      # separate danda symbols
      # sentence = re.sub(r"\|\|", " || ", sentence)
      # sentence = re.sub(r"\|", " | ", sentence)

      # cleanup spaces again
      sentence = re.sub(r"\s+", " ", sentence)

      """
      The following are implemented after initial encoding <==> decoding
      comparision
      """

      # cleaning nukta consonants to base consonants
      char_map = {
          "क़": "क",
          "ख़": "ख",
          "ग़": "ग",
          "ज़": "ज",
          "फ़": "फ",
          "ऩ": "न",
          "ऱ": "र"}

      for old, new in char_map.items():
        sentence = sentence.replace(old, new)


      #  cleanup symbols
      sentence = re.sub(r"\s+", " ", sentence)



      texts.append(sentence)

    return texts

In [ ]:
data = clean_text(data)
data[:2]

In [ ]:
# Corpus Statistics:


def corpus_statistics(data):
  """
  corpus statistics
  """

# Calculate metrics for the dataset
  num_verses = len(data)

# Calculate num_lines by counting ' । ' and ' ॥ ' delimiters
  num_lines = 0
  for sentence in data:
    num_lines += sentence.count(' । ')
    num_lines += sentence.count(' ॥ ')


  total_characters = sum(len(sentence.replace(' ', '')) for sentence in data)
  total_words = sum(len(sentence.split()) for sentence in data)

  print(f"Number of verses: {num_verses}")
  print(f"Number of lines: {num_lines}")
  print(f"Total characters (excluding spaces): {total_characters}")
  print(f"Total words: {total_words}")

  # Calculate averages
  average_words_per_verse = total_words / num_verses
  average_chars_per_verse = total_characters / num_verses
  average_words_per_line = total_words / num_lines
  average_chars_per_line = total_characters / num_lines

  print(f"\n")
  print(f"Average words per verse: {average_words_per_verse:.2f}")
  print(f"Average characters per verse: {average_chars_per_verse:.2f}")
  print(f"Average words per line: {average_words_per_line:.2f}")
  print(f"Average characters per line: {average_chars_per_line:.2f}")

In [ ]:
corpus_statistics(data)

## Loading the tokenizers

### GPT Tokenizers for old and new

In [ ]:
# they live in the tiktoken environmemt, and here i can use the name as tokenizers using .encode:

import tiktoken
gpt_enc = tiktoken.get_encoding("cl100k_base")
o200k_enc = tiktoken.get_encoding("o200k_base")

### SentencePiece

As used by (Kumar, 2026)

In [ ]:
import requests
!wget https://raw.githubusercontent.com/NikhilaGadge/Sanskrit_Tokenization_Study/main/sentencepiece/spm_sa.model

In [ ]:
import sentencepiece as spm
from pathlib import Path

def load_spm(lang: str) -> spm.SentencePieceProcessor:
    sp = spm.SentencePieceProcessor()
    sp.load("spm_sa.model")
    return sp

In [ ]:
sp_sa = load_spm("sa")

### SansGPT Tokenizer

In [ ]:
!wget https://raw.githubusercontent.com/rhugvedd/SansGPT-Advancing-Generative-Pre-Training-in-Sanskrit/master/BPETokenizer.py
!wget https://raw.githubusercontent.com/NikhilaGadge/Sanskrit_Tokenization_Study/main/SansGPT/Final-Corpus-Tokenizer-Merge-Info-NL-12000-.pkl
!wget https://raw.githubusercontent.com/NikhilaGadge/Sanskrit_Tokenization_Study/main/SansGPT/Final-Corpus-Tokenizer-Vocab-NL-12000-.pkl

In [ ]:
vocab_path = "./"
merge_info_name = "Final-Corpus-Tokenizer-Merge-Info-NL-12000-"
vocab_name = "Final-Corpus-Tokenizer-Vocab-NL-12000-"

In [ ]:
# SansGPT BPE Tokenizer:
# as implemented in paper

from BPETokenizer import BPETokenizer

Tokenizer = BPETokenizer()
Tokenizer.load(vocab_path, merge_info_name, vocab_name)

In [ ]:
dir(Tokenizer)

In [ ]:
class SansGPTTokeinzer:
    """
    Thin adapter that makes BPETokenizer speak the same interface as every
    other tokenizer in the pipeline:

        .encode(sentence)   → flat list of int token IDs
        .tokenize(sentence) → list of token strings
        .decode(token_ids)  → decoded UTF-8 string

    The pipeline's tokenize() function calls these three methods.
    """

    def __init__(self, tok):
        self.tok = tok
        print(type(tok))

        # Metadata for benchmark tables

        self.vocab_size = len(tok.Vocab)
        self.name_or_path = tok.__class__.__name__

        self.backend_model = type(tok).__name__

        try:
          self.special_tokens_map = {
              "special_tokens": tok.special_tokens
              }
        except:
          self.special_tokens_map = {}





        # --- pre-build DecodedBytes lookup once, reuse across all calls ---
        # Rebuilding this per-call (as Decode() does internally) is O(vocab_size)
        # and would be expensive inside a sentence loop.


        self._decoded_bytes = {}
        for i in range(256):
            self._decoded_bytes[i] = bytes([i])
        for (m0, m1), mgd in self.tok.MergeInfo.items():
            self._decoded_bytes[mgd] = self._decoded_bytes[m0] + self._decoded_bytes[m1]

        print(f"[SansGPTTokenizerWrapper] Loaded. Vocab size: {len(self.tok.Vocab)}, "
              f"MergeInfo entries: {len(self.tok.MergeInfo)}")

    # ------------------------------------------------------------------
    # .encode(sentence) → flat list[int]
    # ------------------------------------------------------------------

    def encode(self, sentence: str) -> list:
        """
        EncodeFromText returns a LIST OF LISTS — one inner list per text
        segment produced by the similarity-word splitter in GetCleanedText.
        We flatten into a single list so the pipeline's tokenize() function
        gets a plain sequence of token IDs.
        """
        # WithoutNewLine=True  → joins lines with spaces (correct for inference)
        # SkipFirstChunkInLine=False → do not strip the first word of each line
        # Replacements={}      → no substitutions at inference time
        nested = self.tok.EncodeFromText(
            sentence,
            WithoutNewLine=False,
            SkipFirstChunkInLine=False,
            Replacements={}
        )
        # Flatten list-of-lists → flat list
        flat = [tok_id for segment in nested for tok_id in segment]
        return flat

    # ------------------------------------------------------------------
    # .tokenize(sentence) → list[str]
    # ------------------------------------------------------------------

    def tokenize(self, sentence: str) -> list:
        """
        Returns each token as a human-readable string.
        Uses the pre-built _decoded_bytes lookup so this is O(n_tokens).
        """
        token_ids = self.encode(sentence)
        token_strings = []
        for tid in token_ids:
            raw_bytes = self._decoded_bytes.get(tid, b'')
            token_strings.append(raw_bytes.decode('utf-8', errors='replace'))
        return token_strings

    # ------------------------------------------------------------------
    # .decode(token_ids) → str
    # ------------------------------------------------------------------

    def decode(self, token_ids: list, **kwargs) -> str:
        """
        Reconstructs a UTF-8 string from a flat list of token IDs.

        The pipeline's tokenize() function tries:
            tokenizer.decode(tokens, skip_special_tokens=True)
        and falls back to:
            tokenizer.decode(tokens)
        **kwargs absorbs skip_special_tokens so both call-sites work without error.
        """
        raw = b''.join(
            self._decoded_bytes.get(tid, b'')
            for tid in token_ids
        )
        return raw.decode('utf-8', errors='replace')

In [ ]:
sansgpt = SansGPTTokeinzer(Tokenizer)

In [ ]:
print(sansgpt.backend_model)

In [ ]:
# Paste and run this cell immediately
print("Vocab size:      ", len(sansgpt.tok.Vocab))
print("MergeInfo size:  ", len(sansgpt.tok.MergeInfo))

# Check a few MergeInfo keys
sample = list(sansgpt.tok.MergeInfo.items())[:5]
print("Sample MergeInfo entries:")
for k, v in sample:
    print(f"  {k} → {v}  | key type: {type(k)}, element types: {type(k[0])}, {type(k[1])}")

### Sutra

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer

login(userdata.get("HF_TOKEN"))

In [ ]:
sutra = AutoTokenizer.from_pretrained("TWO/sutra-mlt256-v2")

### HF tokenizers

In [ ]:
from transformers import AutoTokenizer

claude = AutoTokenizer.from_pretrained("Xenova/claude-tokenizer")
tiny = AutoTokenizer.from_pretrained("openaccess-ai-collective/tiny-mistral")
sutra = AutoTokenizer.from_pretrained("TWO/sutra-mlt256-v2")
qwen_sanskrit_model = AutoTokenizer.from_pretrained("diabolic6045/Sanskrit-Qwen2.5-7B-base")
qwen_sanskrit_tokenizer = AutoTokenizer.from_pretrained("diabolic6045/Sanskrit-English-qwen2-tokenizer")
mt5 = AutoTokenizer.from_pretrained("google/mt5-small")
mbart = AutoTokenizer.from_pretrained("facebook/mbart-large-50")
airavata = AutoTokenizer.from_pretrained("ai4bharat/Airavata") # needs login
llama = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct") # needs login # access granted
aya = AutoTokenizer.from_pretrained("CohereLabs/aya-expanse-8b") # needs login
gemma = AutoTokenizer.from_pretrained("google/gemma-1.1-2b-it") # needs login

## Metadata Tokenizers

In [ ]:
import pandas as pd


def tokenizer_metadata_table(tokenizers_dict):

    rows = []

    for label, tok in tokenizers_dict.items():

        row = {
            "label": label,
            "class": getattr(tok, "backend_model", type(tok).__name__),
        }

        # -----------------------------
        # vocab size
        # -----------------------------

        if hasattr(tok, "vocab_size"):
          vocab = tok.vocab_size

          if callable(vocab):
            row["vocab_size"] = vocab()
          else:
            row["vocab_size"] = vocab

        elif hasattr(tok, "n_vocab"):

          row["vocab_size"] = tok.n_vocab

        else:

          row["vocab_size"] = "not available"

        # -----------------------------
        # model path / tokenizer name
        # -----------------------------
        if hasattr(tok, "name_or_path"):

            row["model_path"] = tok.name_or_path

        elif hasattr(tok, "name"):

            row["model_path"] = tok.name

        else:

            row["model_path"] = "not available"


        # -----------------------------
        # backend tokenizer model
        # -----------------------------
        try:

            row["backend_model"] = str(tok.backend_tokenizer.model)

        except:
          if hasattr(tok, "backend_model"):
            row["backend_model"] = tok.backend_model

          else:
            row["backend_model"] = tok.__class__.__name__


        # -----------------------------
        # special tokens
        # -----------------------------
        try:

            row["special_tokens"] = tok.special_tokens_map

        except:

            try:

                row["special_tokens"] = tok._special_tokens

            except:

                row["special_tokens"] = "not available"

        # -----------------------------
        # sentencepiece model
        # -----------------------------

        if type(tok).__name__ == "SentencePieceProcessor":
          row["model_path"] = getattr(tok, "_model_file", "SentencePiece Model")
          row["backend_model"] = type(tok).__name__

          try:
            row["special_tokens"] = {
                "unk_id": tok.unk_id(),
                "bos_id": tok.bos_id(),
                "eos_id": tok.eos_id(),
                "pad_id": tok.pad_id()
                }
          except:
            row["special_tokens"] = "not available"

        rows.append(row)

    df = pd.DataFrame(rows)

    return df

## Evaluation

In [ ]:
tokenizers = {

    "claude": claude,
    "tiny_mistral": tiny,
    "sutra": sutra,
    "qwen_sanskrit_model": qwen_sanskrit_model,
    "qwen_sanskrit_tokenizer": qwen_sanskrit_tokenizer,
    "mt5": mt5,
    "mbart": mbart,
    "airavata" : airavata, # needs login
    "llama": llama, # needs login # access granted
    "aya": aya, # needs login
    "gemma": gemma, # needs login
    "cl100k_base": gpt_enc,
    "o200k_base": o200k_enc,
    "sentencepiece_sa": sp_sa,
    "sansgpt_tokenizer": sansgpt
}

meta_df = tokenizer_metadata_table(tokenizers)

meta_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sort meta_df by vocab_size in ascending order
sorted_meta_df = meta_df.sort_values(by='vocab_size', ascending=True)

plt.figure(figsize=(8, 4))
sns.barplot(x='label', y='vocab_size', data=sorted_meta_df, palette='viridis', hue='label', legend=False, order=sorted_meta_df['label'])
# Add a rising red line
sns.lineplot(x='label', y='vocab_size', data=sorted_meta_df, color='red', marker='o', linewidth=2, sort=False)
plt.title('Vocabulary Size of Different Tokenizers (Ascending)')
plt.xlabel('Tokenizer Label')
plt.ylabel('Vocabulary Size')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

def tokenize(data: list, tokenizer, tokenizer_label):
    """
    Tokenizes each sentence individually.
    Returns a DataFrame — one row per sentence.
    """

    records = []

    for sentence in data:

        if not sentence.strip():
            continue

        tokens = tokenizer.encode(sentence)

        try:
          token_strings = tokenizer.tokenize(sentence)
        except:
          token_strings = [tokenizer.decode([t]) for t in tokens] # Converts each token ID back into readable token text/subword like 15266 --> Hello

        words = sentence.split()

        num_tokens = len(tokens) # Number of total generated tokens per sentence
        num_words = len(words) # number of total words in sentence
        num_chars = len(sentence.replace(" ", "")) # total number of charachters --> in english one letter - one charachter

        try:
            decoded = tokenizer.decode(tokens, skip_special_tokens=True)
        except:
            decoded = tokenizer.decode(tokens)

        records.append({

            "sentence": sentence,
            "decoded_sentence": decoded,
            "tokens": token_strings,
            "similarity": sentence.strip() == decoded.strip(),
            "num_words": num_words,
            "num_tokens": num_tokens,
            # "unique_tokens_count": len(set(tokens)), # more useful at corpus level
            "token_ids": tokens,
            "num_chars": num_chars,
            "fertility": num_tokens / num_words if num_words > 0 else None, # Average tokens per word
            "tokens_per_char": num_tokens / num_chars if num_chars > 0 else None, # how many tokens needed to represent one charachter
            "chars_per_token": num_chars / num_tokens if num_tokens > 0 else None, # aka Compression Ratio , how many chars packed into one token?
            "tokenizer": tokenizer_label
        })

    df = pd.DataFrame(records)

    #filename = f"tokenizer_analysis_{tokenizer_label}.xlsx"

    #df.to_excel(filename, index=False)

    #files.download(filename)

    return df

In [ ]:
test = data[0]
print(test)
ids = sansgpt.encode(test)

print(ids)

#print(Tokenizer.Decode(ids[:10]))
print(Tokenizer.Decode(ids))

import inspect

print(inspect.getsource(BPETokenizer.Decode))
help(Tokenizer.Decode)

In [ ]:
test = data[0]
print(test)
ids = sansgpt.encode(test)

print(ids)
print(type(tokens))
print(tokens)
len(tokens)

In [ ]:
test = data[0]
print(test)

tokens = sansgpt.encode(test)
print("\n")
print(tokens)
print(type(tokens))
print(tokens)
print(len(tokens))


Tokens = Tokenizer.EncodeFromText(
    test,
    WithoutNewLine=False,
    SkipFirstChunkInLine=False,
    Replacements={}
)

print("\n")
print(Tokens)
print(type(Tokens))
print(Tokens)
print(len(Tokens))

In [ ]:
test = data[0]

r1 = Tokenizer.EncodeFromText(test, WithoutNewLine=True,  SkipFirstChunkInLine=False, Replacements={})
r2 = Tokenizer.EncodeFromText(test, WithoutNewLine=False, SkipFirstChunkInLine=False, Replacements={})

flat1 = [t for seg in r1 for t in seg]
flat2 = [t for seg in r2 for t in seg]

print("WithoutNewLine=True  token count:", len(flat1))
print("WithoutNewLine=False token count:", len(flat2))
print("Identical?", flat1 == flat2)

In [ ]:
for chunk in Tokens[:10]:
    print(chunk)
    print(Tokenizer.Decode(chunk))
    print("------")

## Tokenizer wise results:

In [ ]:
tokenize(data, qwen_sanskrit_tokenizer,"qwen_sanskrit_tokenizer").head(2)

In [ ]:
tokenize(data, claude,"claude").head(2)

In [ ]:
tokenize(data, gpt_enc,"cl100k_base").head(2)

In [ ]:
tokenizer_names = []

tokenizers = [

    ("claude", claude),
    ("tiny", tiny),
    ("sutra", sutra),
    ("qwen_sanskrit_model", qwen_sanskrit_model),
    ("qwen_sanskrit_tokenizer", qwen_sanskrit_tokenizer),
    ("mt5", mt5),
    #("mbart", mbart),
    ("airavata" , airavata), # needs login
    ("llama", llama), # needs login # access granted
    ("aya", aya), # needs login
    ("gemma", gemma), # needs login
    ("cl100k_base", gpt_enc),
    ("o200k_base", o200k_enc),
    ("sentencepiece_sa", sp_sa),
    ("sansgpt_tokenizer", sansgpt)

]

for tok, name in tokenizers:
  #print(f"Running {name}")
  temp_df = tokenize(data, name, tok)
  tokenizer_names.append(temp_df)

df = pd.concat(tokenizer_names, ignore_index = True)

In [ ]:
# uncomment to download a file

"""

filename = f"tokenizer_analysis.xlsx"

df.to_excel(filename, index=False)

files.download(filename)

"""

In [ ]:
df['tokenizer'].unique()

In [ ]:
df.head(2)

In [ ]:
df.tail(2)

In [ ]:
df[df['sentence'] == "तस्यां चीरं वसानायां नाथवत्यामनाथवत् । प्रचुक्रोश जनः सर्वो धिक् त्वां दशरथं त्विति ॥ "]

### Encoding and Decoding Similarity:

In [ ]:
df['similarity'].value_counts()

In [ ]:
df[df['similarity'] == False][['similarity','tokenizer']].drop_duplicates()

## Summary of Metrices

In [ ]:
summary = (
    df
    .groupby("tokenizer")
    .agg({
        "num_words": "mean",
        "num_tokens": "mean",
        "fertility": "mean",
        "tokens_per_char": "mean",
        "chars_per_token": "mean"
    })
    .sort_values("fertility", ascending=True)
)
summary_round = round(summary, 1)
summary_round

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate average fertility per tokenizer
fertility_summary = (
    df.groupby('tokenizer')['fertility']
      .mean()
      .reset_index()
      .sort_values('fertility', ascending=True)
)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='tokenizer',
    y='fertility',
    data=fertility_summary,
    palette='viridis'
)

plt.title('Average Fertility by Tokenizer')
plt.xlabel('Tokenizer')
plt.ylabel('Average Fertility (Tokens per Word)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Conclusion

# Vedic Corpus

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset
vedic = load_dataset("shunyasea/vedic-sanskrit")

In [ ]:
print(vedic)

In [ ]:
vedic = vedic["test"]["text"]

In [ ]:
vedic[:5]

In [ ]:
len(vedic)

In [ ]:
vedic = clean_text(vedic)

In [ ]:
corpus_statistics(vedic)

In [ ]:
tokenizer_names = []

tokenizers = [

    ("claude", claude),
    ("tiny", tiny),
    ("sutra", sutra),
    ("qwen_sanskrit_model", qwen_sanskrit_model),
    ("qwen_sanskrit_tokenizer", qwen_sanskrit_tokenizer),
    ("mt5", mt5),
    ("mbart", mbart),
    ("airavata" , airavata), # needs login
    ("llama", llama), # needs login # access granted
    ("aya", aya), # needs login
    ("gemma", gemma), # needs login
    ("cl100k_base", gpt_enc),
    ("o200k_base", o200k_enc),


]

for tok, name in tokenizers:
  temp_df = tokenize(vedic, name, tok)
  tokenizer_names.append(temp_df)

df = pd.concat(tokenizer_names, ignore_index = True)

In [ ]:
summary = (
    df
    .groupby("tokenizer")
    .agg({
        "num_words": "mean",
        "num_tokens": "mean",
        "fertility": "mean",
        "tokens_per_char": "mean",
        "chars_per_token": "mean"
    })
    .sort_values("fertility", ascending=True)
)
summary_round = round(summary, 1)
summary_round

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate average fertility per tokenizer
fertility_summary = (
    df.groupby('tokenizer')['fertility']
      .mean()
      .reset_index()
      .sort_values('fertility', ascending=True)
)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='tokenizer',
    y='fertility',
    data=fertility_summary,
    palette='viridis'
)

plt.title('Average Fertility by Tokenizer')
plt.xlabel('Tokenizer')
plt.ylabel('Average Fertility (Tokens per Word)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()